# HUST CQRF+CBR / QRR+CBR — Paired Protocol, Manuscript Eq. 12 Features (Corrected v1)

This notebook corrects the earlier `346_..._v1` template: the input features are now the
**submitted manuscript's Eq. 12 set**, extracted with the **verbatim v2 extraction code**:

`[voltage_mean, voltage_median, duration, abs_current_mean, Qt, voltage_slope]` (Qt at zero-based index 4)

It runs **two models per seed** (CQRF+CBR and QRR+CBR, no No-Qt third model), so a seed costs
about two-thirds of a full v2 seed (~4.2 h instead of ~6.3 h on a Colab CPU).

## Compatibility with the v2 strict-reference notebook

Feature extraction (column resolution, discharge-phase filter, time-sorted duration and
least-squares voltage slope, `Q0 = median of first five fully feature-valid capacities`,
`0 < SOH < 1.5` target filter), the frozen 77-cell universe, the lexicographic 60/20/20
splits, the shared `RandomState(42)` reservoir, the per-cell warmup, the train-only raw
scaler, and the 125-parameter Tanh CBR are all copied from the v2 notebook, so for the same
seed this notebook is designed to reproduce v2's CQRF+CBR and QRR+CBR rows.

Because of that, this notebook uses the **same `PROTOCOL_TAG` as v2** and its summary cell
merges v2's completed `HUST_per_seed_metrics.csv` rows with this notebook's rows. Seeds that
v2 already finished are skipped automatically when v2's experiment ID is listed in
`PRIOR_EXP_IDS`.

## Parallel execution

Make copies and change only `SEEDS` and `EXP_ID`, for example `[0, 1, 2]`, `[3, 4, 5]`,
`[6, 7]`, `[8, 9]`. Keep every scientific setting, the cell-ID list, and `PROTOCOL_TAG`
identical. List every other copy (and the v2 experiment ID) in `PRIOR_EXP_IDS` of the copy
you use for the final 10-seed summary and Wilcoxon test.


In [ ]:
# ================================================================
# 1. Environment and paths
# ================================================================
import os, sys, subprocess, time, json, warnings, pickle, datetime, copy, glob, random, gc, math, re, hashlib
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple


def _ensure_packages():
    required = {
        'pennylane': 'pennylane',
        'scipy': 'scipy',
        'psutil': 'psutil',
        'sklearn': 'scikit-learn',
    }
    missing = []
    for module_name, package_name in required.items():
        try:
            __import__(module_name)
        except Exception:
            missing.append(package_name)
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


_ensure_packages()

import numpy as np
import pandas as pd
import pennylane as qml
import psutil

from scipy.stats import wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception:
    pass

PROJECT_ROOT = '/content/drive/MyDrive/HB/QRC_Battery_SOH'

HUST_CANDIDATE_PATHS = [
    os.path.join(PROJECT_ROOT, 'data/raw/HUST'),
    os.path.join(PROJECT_ROOT, 'data/processed/HUST'),
    '/content/drive/MyDrive/HUST',
    '/content/drive/MyDrive/HB/HUST_processed',
    '/content/HUST',
    '/mnt/data/HUST',
]
HUST_PATH = next((p for p in HUST_CANDIDATE_PATHS if os.path.exists(p)), HUST_CANDIDATE_PATHS[0])

# Change this in each parallel copy.
EXP_ID = 'v3_C89_HUST_CQRF_QRR_CBR_PAIRED_MANUSCRIPT6_v1'

# Same tag as the v2 strict-reference notebook: extraction, splits, reservoir,
# warmup, scaler, and CBR are verbatim-compatible, so rows are mergeable.
PROTOCOL_TAG = 'paired_v2_hust_manuscript6_mit_strict_reference'

# Metrics from earlier compatible runs. The v2 notebook's experiment ID belongs
# here so its completed CQRF/QRR seeds are skipped and merged in the summary.
PRIOR_EXP_IDS = [
    'EXP-HUST-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-MIT-STRICT-REFERENCE-v2',
    # '347_HUST-CQRF-QRR-CBR-PAIRED-MANUSCRIPT6-SEEDS-3-5-v1',
    # '347_HUST-CQRF-QRR-CBR-PAIRED-MANUSCRIPT6-SEEDS-6-7-v1',
    # '347_HUST-CQRF-QRR-CBR-PAIRED-MANUSCRIPT6-SEEDS-8-9-v1',
]
# Both metric filenames are searched per experiment ID (v2 used the first).
PRIOR_METRIC_FILENAMES = ['HUST_per_seed_metrics.csv', 'HUST_paired_run_metrics.csv']

CKPT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', EXP_ID)
RES_DIR = os.path.join(PROJECT_ROOT, 'results', EXP_ID)
LOG_DIR = os.path.join(PROJECT_ROOT, 'logs', EXP_ID)
FEAT_DIR = os.path.join(PROJECT_ROOT, 'features', EXP_ID)
SPLIT_DIR = os.path.join(RES_DIR, 'splits')
PRED_DIR = os.path.join(RES_DIR, 'predictions')
HIST_DIR = os.path.join(RES_DIR, 'histories')

for directory in [CKPT_DIR, RES_DIR, LOG_DIR, FEAT_DIR, SPLIT_DIR, PRED_DIR, HIST_DIR]:
    os.makedirs(directory, exist_ok=True)

LOG_PATH = os.path.join(LOG_DIR, f'{EXP_ID}.log')


def log(message):
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{timestamp}] {message}'
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as handle:
        handle.write(line + '\n')


print('PROJECT_ROOT:', PROJECT_ROOT)
print('HUST_PATH:', HUST_PATH, 'exists=', os.path.exists(HUST_PATH))
print('EXP_ID:', EXP_ID)
print('Results:', RES_DIR)


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/HB/QRC_Battery_SOH
HUST_PATH: /content/drive/MyDrive/HB/QRC_Battery_SOH/data/raw/HUST exists= True
EXP_ID: v3_C89_HUST_CQRF_QRR_CBR_PAIRED_MANUSCRIPT6_v1
Results: /content/drive/MyDrive/HB/QRC_Battery_SOH/results/v3_C89_HUST_CQRF_QRR_CBR_PAIRED_MANUSCRIPT6_v1


In [ ]:
# ================================================================
# 2. Frozen paired-protocol configuration (manuscript Eq. 12 features)
# ================================================================
RUN_MODE = 'FULL'      # SMOKE uses SMOKE_SEEDS. FULL uses SEEDS.
SEEDS = [8, 9]
SMOKE_SEEDS = [0]

# Manuscript Eq. 12 feature order. Qt is zero-based index 4.
WITH_QT_FEATURE_NAMES = [
    'voltage_mean',
    'voltage_median',
    'duration',
    'abs_current_mean',
    'Qt',
    'voltage_slope',
]
FULL_INPUT_DIM = 6
QT_COLUMN_INDEX = 4
HUST_FEATURE_NAMES = WITH_QT_FEATURE_NAMES

# Frozen verified HUST cell universe (from the approved v2 audit).
EXPECTED_CELL_IDS = [
    '1-1', '1-2', '1-3', '1-4', '1-5', '1-6', '1-7', '1-8',
    '2-2', '2-3', '2-4', '2-5', '2-6', '2-7', '2-8',
    '3-1', '3-2', '3-3', '3-4', '3-5', '3-6', '3-7', '3-8',
    '4-1', '4-2', '4-3', '4-4', '4-5', '4-6', '4-7', '4-8',
    '5-1', '5-2', '5-3', '5-4', '5-5', '5-6', '5-7',
    '6-1', '6-2', '6-3', '6-4', '6-5', '6-6', '6-8',
    '7-1', '7-2', '7-3', '7-4', '7-5', '7-6', '7-7', '7-8',
    '8-1', '8-2', '8-3', '8-4', '8-5', '8-6', '8-7', '8-8',
    '9-1', '9-2', '9-3', '9-4', '9-5', '9-6', '9-7', '9-8',
    '10-1', '10-2', '10-3', '10-4', '10-5', '10-6', '10-7', '10-8',
]

# HUST loader controls (verbatim v2 semantics).
COLUMN_OVERRIDES = {'voltage': None, 'current': None, 'time': None, 'phase': None}
ALLOW_SAMPLE_INDEX_AS_TIME = False
QT_SCALE = 1.0

WARMUP = 5
RESERVOIR_SEED = 42
COUPLING_STRENGTH = 0.3
N_MEMORY = 4
N_PROCESSOR = 6
N_RESERVOIR_LAYERS = 2

USE_FEATURE_CACHE = True
REGENERATE_FEATURES = False
SKIP_COMPLETED = True

CBR_HIDDEN_DIMS = (2, 2)
CBR_DROPOUT = 0.0
CBR_EPOCHS = 300
CBR_LR = 1e-3
CBR_BATCH_SIZE = 64
CBR_PATIENCE = 40
CBR_WEIGHT_DECAY = 1e-5
CBR_GRAD_CLIP = 1.0

REFERENCE_MIT_CONFIG = {
    'WARMUP': 5,
    'RESERVOIR_SEED': 42,
    'COUPLING_STRENGTH': 0.3,
    'N_MEMORY': 4,
    'N_PROCESSOR': 6,
    'N_RESERVOIR_LAYERS': 2,
    'CBR_HIDDEN_DIMS': (2, 2),
    'CBR_DROPOUT': 0.0,
    'CBR_EPOCHS': 300,
    'CBR_LR': 1e-3,
    'CBR_BATCH_SIZE': 64,
    'CBR_PATIENCE': 40,
    'CBR_WEIGHT_DECAY': 1e-5,
    'CBR_GRAD_CLIP': 1.0,
}
CURRENT_CONFIG = {
    'WARMUP': WARMUP,
    'RESERVOIR_SEED': RESERVOIR_SEED,
    'COUPLING_STRENGTH': COUPLING_STRENGTH,
    'N_MEMORY': N_MEMORY,
    'N_PROCESSOR': N_PROCESSOR,
    'N_RESERVOIR_LAYERS': N_RESERVOIR_LAYERS,
    'CBR_HIDDEN_DIMS': CBR_HIDDEN_DIMS,
    'CBR_DROPOUT': CBR_DROPOUT,
    'CBR_EPOCHS': CBR_EPOCHS,
    'CBR_LR': CBR_LR,
    'CBR_BATCH_SIZE': CBR_BATCH_SIZE,
    'CBR_PATIENCE': CBR_PATIENCE,
    'CBR_WEIGHT_DECAY': CBR_WEIGHT_DECAY,
    'CBR_GRAD_CLIP': CBR_GRAD_CLIP,
}
assert CURRENT_CONFIG == REFERENCE_MIT_CONFIG

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def count_dense_params(input_dim, hidden_dims):
    dims = [input_dim, *hidden_dims, 1]
    return sum(dims[i] * dims[i + 1] + dims[i + 1] for i in range(len(dims) - 1))


EXPECTED_CBR_PARAMS = count_dense_params(57, CBR_HIDDEN_DIMS)
assert EXPECTED_CBR_PARAMS == 125
assert FULL_INPUT_DIM == 6 and QT_COLUMN_INDEX == 4
assert WITH_QT_FEATURE_NAMES[QT_COLUMN_INDEX] == 'Qt'
assert len(EXPECTED_CELL_IDS) == 77

EXPECTED_INPUT_DIM = FULL_INPUT_DIM
ACTIVE_SEEDS = SMOKE_SEEDS if RUN_MODE.upper() == 'SMOKE' else SEEDS

MODEL_LABEL_CQRF = 'CQRF+CBR'
MODEL_LABEL_QRR = 'QRR+CBR'
MODEL_LABELS = [MODEL_LABEL_CQRF, MODEL_LABEL_QRR]

print('RUN_MODE:', RUN_MODE)
print('ACTIVE_SEEDS:', ACTIVE_SEEDS)
print('Device:', device)
print('Models per seed:', MODEL_LABELS, '(No-Qt intentionally excluded for runtime)')
print('Manuscript Eq. 12 inputs:', WITH_QT_FEATURE_NAMES)
print('Shared CBR: 57 -> 2 -> 2 -> 1, Tanh, dropout=0.0, params=', EXPECTED_CBR_PARAMS)


RUN_MODE: FULL
ACTIVE_SEEDS: [8, 9]
Device: cpu
Models per seed: ['CQRF+CBR', 'QRR+CBR'] (No-Qt intentionally excluded for runtime)
Manuscript Eq. 12 inputs: ['voltage_mean', 'voltage_median', 'duration', 'abs_current_mean', 'Qt', 'voltage_slope']
Shared CBR: 57 -> 2 -> 2 -> 1, Tanh, dropout=0.0, params= 125


In [ ]:
# ================================================================
# 3. Paired 10-qubit reservoirs with shared frozen parameters
#    - CQRF: memory register reset every cycle (non-recurrent)
#    - QRR:  memory register recurrently updated from Z measurements
# ================================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


class PairedReservoir10q:
    """Fixed 10-qubit reservoir shared by CQRF (non-recurrent) and QRR (recurrent).

    Frozen parameters are drawn from a single RNG stream in the same order as
    the completed MIT template:
      1) reservoir_params  (layers, qubits, 2)
      2) input_projection  (n_inputs, n_processor), randn * 0.5
      3) memory_rotation   (n_memory, 2)
    The circuit returns 61 expectation values: the 57 reservoir features
    followed by the 4 memory-register Z values used for the QRR update.
    """

    def __init__(
        self,
        n_inputs,
        recurrent,
        n_memory=N_MEMORY,
        n_processor=N_PROCESSOR,
        n_reservoir_layers=N_RESERVOIR_LAYERS,
        seed=RESERVOIR_SEED,
        coupling_strength=COUPLING_STRENGTH,
    ):
        self.n_memory = int(n_memory)
        self.n_processor = int(n_processor)
        self.n_qubits = self.n_memory + self.n_processor
        self.n_reservoir_layers = int(n_reservoir_layers)
        self.n_inputs = int(n_inputs)
        self.seed = int(seed)
        self.coupling_strength = float(coupling_strength)
        self.recurrent = bool(recurrent)

        assert self.n_inputs == EXPECTED_INPUT_DIM == 6

        self.memory_wires = list(range(self.n_memory))
        self.proc_wires = list(range(self.n_memory, self.n_qubits))
        self.proc_pairs = [(i, j) for i in self.proc_wires for j in self.proc_wires if i < j]
        self.cross_pairs = [(i, j) for i in self.memory_wires for j in self.proc_wires]

        self.n_single = self.n_processor * 3
        self.n_proc_pairs = len(self.proc_pairs)
        self.n_cross_pairs = len(self.cross_pairs)
        self.n_outputs = self.n_single + self.n_proc_pairs + self.n_cross_pairs
        assert self.n_outputs == 57

        rng = np.random.RandomState(self.seed)
        self.reservoir_params = rng.uniform(
            0, 2 * np.pi, size=(self.n_reservoir_layers, self.n_qubits, 2)
        )
        self.input_projection = rng.randn(self.n_inputs, self.n_processor) * 0.5
        self.memory_rotation = rng.uniform(0, 2 * np.pi, size=(self.n_memory, 2))

        self.dev = qml.device('default.qubit', wires=self.n_qubits)
        self.memory_angles = np.zeros(self.n_memory, dtype=np.float64)
        self._build_circuit()

    @property
    def model_name(self):
        return MODEL_LABEL_QRR if self.recurrent else MODEL_LABEL_CQRF

    def _build_circuit(self):
        n_q = self.n_qubits
        n_m = self.n_memory
        n_p = self.n_processor
        n_l = self.n_reservoir_layers
        rp = self.reservoir_params
        mw = self.memory_wires
        pw = self.proc_wires
        pp = self.proc_pairs
        cp = self.cross_pairs
        coupling = self.coupling_strength
        memory_rotation = self.memory_rotation

        @qml.qnode(self.dev)
        def circuit(projected_input, memory_angles):
            for i, wire in enumerate(mw):
                qml.RY(memory_angles[i], wires=wire)
                qml.RZ(memory_rotation[i, 0], wires=wire)

            for i, wire in enumerate(pw):
                qml.RY(projected_input[i], wires=wire)
                qml.RZ(projected_input[i] * 0.5, wires=wire)

            for i in range(n_m - 1):
                qml.CNOT(wires=[mw[i], mw[i + 1]])
            for i in range(n_p - 1):
                qml.CNOT(wires=[pw[i], pw[i + 1]])
            qml.CNOT(wires=[pw[-1], pw[0]])

            for i in range(n_m):
                qml.CRY(coupling * np.pi, wires=[mw[i], pw[i]])
                qml.CRY(coupling * 0.5 * np.pi, wires=[pw[i], mw[i]])
            for i in range(n_m):
                qml.CRY(coupling * 0.3 * np.pi, wires=[mw[i], pw[(i + 2) % n_p]])

            for layer in range(n_l):
                for i in range(n_q):
                    qml.RY(rp[layer, i, 0], wires=i)
                    qml.RZ(rp[layer, i, 1], wires=i)
                for i in range(n_q):
                    qml.CNOT(wires=[i, (i + 1) % n_q])
                for i in range(0, n_q - 2, 2):
                    qml.CNOT(wires=[i, i + 2])

            observables = []
            for wire in pw:
                observables.extend([
                    qml.expval(qml.PauliX(wire)),
                    qml.expval(qml.PauliY(wire)),
                    qml.expval(qml.PauliZ(wire)),
                ])
            for i, j in pp:
                observables.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)))
            for i, j in cp:
                observables.append(qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)))
            for wire in mw:
                observables.append(qml.expval(qml.PauliZ(wire)))
            return observables

        self._circuit = circuit

    def reset(self):
        self.memory_angles.fill(0.0)

    def process_single(self, x):
        projected_input = np.clip(x @ self.input_projection, -np.pi, np.pi)
        outputs = np.asarray(self._circuit(projected_input, self.memory_angles), dtype=np.float64)
        if self.recurrent:
            memory_z = outputs[self.n_outputs : self.n_outputs + self.n_memory]
            self.memory_angles = (memory_z + 1.0) * np.pi
        else:
            self.memory_angles.fill(0.0)
        return outputs[:self.n_outputs].astype(np.float32)

    def process_cell(self, x_cell):
        self.reset()
        states = np.zeros((len(x_cell), self.n_outputs), dtype=np.float32)
        for index, x in enumerate(x_cell):
            states[index] = self.process_single(x)
        return states


def normalize_raw_features(features, stats):
    normalized = (features - stats['mean']) / stats['std']
    return np.clip(normalized, -3, 3) * (np.pi / 3)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None)))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        'MAPE': float(mape),
        'RMSE': float(rmse),
        'MAE': float(mae),
        'R2': float(r2),
        'DerivedScore': float((1.0 - mape) * 100.0),
    }


def current_rss_mb():
    return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))


In [ ]:
# ================================================================
# 4. HUST loader — verbatim v2 extraction, manuscript Eq. 12 features
#    [voltage_mean, voltage_median, duration, abs_current_mean, Qt, voltage_slope]
# ================================================================
def natural_key(value):
    return tuple(int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', str(value)))


def to_scalar(value):
    try:
        if isinstance(value, (int, float, np.generic)):
            return float(value)
        arr = np.asarray(value).reshape(-1)
        return float(arr[0]) if arr.size else float('nan')
    except Exception:
        return float('nan')


def norm_name(name):
    return ''.join(ch.lower() for ch in str(name) if ch.isalnum())


EXACT_NAMES = {
    'voltage': {'voltage', 'voltagev', 'cellvoltage', 'terminalvoltage', 'volt'},
    'current': {'current', 'currenta', 'cellcurrent', 'loadcurrent', 'curr'},
    'time': {'time', 'times', 'timestamp', 'elapsedtime', 'testtime', 'steptime'},
    'phase': {'phase', 'status', 'state', 'mode', 'type', 'operation', 'step'},
}
TOKENS = {
    'voltage': ('voltage', 'volt'),
    'current': ('current', 'curr'),
    'time': ('time', 'timestamp', 'elapsed'),
    'phase': ('phase', 'status', 'state', 'mode', 'type', 'operation', 'step'),
}


def resolve_column(df, role, numeric):
    override = COLUMN_OVERRIDES.get(role)
    if override is not None:
        if override not in df.columns:
            raise KeyError(f'Configured {role} column {override!r} not found in {list(df.columns)}')
        return override, 'manual_override'

    candidates = list(df.columns)
    if numeric:
        candidates = [c for c in candidates if pd.to_numeric(df[c], errors='coerce').notna().sum() >= 2]
    normalized = {c: norm_name(c) for c in candidates}

    exact = [c for c, n in normalized.items() if n in EXACT_NAMES[role]]
    if len(exact) == 1:
        return exact[0], 'exact_name'
    if len(exact) > 1:
        return None, f'ambiguous_exact:{list(map(str, exact))}'

    token = [c for c, n in normalized.items() if any(t in n for t in TOKENS[role])]
    if len(token) == 1:
        return token[0], 'token_match'
    if len(token) > 1:
        return None, f'ambiguous_token:{list(map(str, token))}'
    return None, 'not_found'


def phase_filter(df):
    info = {'phase_policy': None}
    phase_col, reason = resolve_column(df, 'phase', numeric=False)
    if phase_col is None:
        if reason.startswith('ambiguous'):
            info['phase_policy'] = 'unverified_ambiguous_phase_column_using_full_dataframe_for_audit'
            return df, info
        info['phase_policy'] = 'unverified_no_phase_column_using_full_dataframe_for_audit'
        return df, info

    values = df[phase_col].astype(str).str.strip()
    discharge_mask = values.str.lower().str.contains('discharg', regex=False)
    if discharge_mask.any():
        info['phase_policy'] = 'filtered_values_containing_discharg'
        return df.loc[discharge_mask].copy(), info

    info['phase_policy'] = 'unverified_no_discharge_label_using_full_dataframe_for_audit'
    return df.copy(), info


def extract_cycle_raw(df, qt):
    """Verbatim v2 manuscript-feature extraction for one cycle dataframe."""
    info = {'phase_policy': None, 'used_sample_index_time': False, 'failure_reason': None}
    if not isinstance(df, pd.DataFrame) or df.empty:
        info['failure_reason'] = 'empty_or_non_dataframe'
        return None, info

    discharge_df, pinfo = phase_filter(df)
    info.update(pinfo)
    if discharge_df is None or discharge_df.empty:
        info['failure_reason'] = pinfo.get('phase_policy', 'phase_filter_failed')
        return None, info

    v_col, v_reason = resolve_column(discharge_df, 'voltage', numeric=True)
    i_col, i_reason = resolve_column(discharge_df, 'current', numeric=True)
    t_col, t_reason = resolve_column(discharge_df, 'time', numeric=True)
    if v_col is None:
        info['failure_reason'] = f'voltage_unresolved:{v_reason}'
        return None, info
    if i_col is None:
        info['failure_reason'] = f'current_unresolved:{i_reason}'
        return None, info

    voltage = pd.to_numeric(discharge_df[v_col], errors='coerce').to_numpy(float)
    current = pd.to_numeric(discharge_df[i_col], errors='coerce').to_numpy(float)
    if t_col is not None:
        t = pd.to_numeric(discharge_df[t_col], errors='coerce').to_numpy(float)
    elif ALLOW_SAMPLE_INDEX_AS_TIME:
        t = np.arange(len(discharge_df), dtype=float)
        info['used_sample_index_time'] = True
    else:
        info['failure_reason'] = f'time_unresolved:{t_reason}'
        return None, info

    n = min(len(voltage), len(current), len(t))
    voltage, current, t = voltage[:n], current[:n], t[:n]
    mask = np.isfinite(voltage) & np.isfinite(current) & np.isfinite(t)
    voltage, current, t = voltage[mask], current[mask], t[mask]
    if len(t) < 2:
        info['failure_reason'] = 'fewer_than_two_joint_finite_samples'
        return None, info

    order = np.argsort(t, kind='stable')
    voltage, current, t = voltage[order], current[order], t[order]
    duration = float(t[-1] - t[0])
    if not np.isfinite(duration) or duration <= 0:
        info['failure_reason'] = 'nonpositive_duration'
        return None, info

    centered_t = t - t.mean()
    denominator = float(np.dot(centered_t, centered_t))
    if denominator <= 0:
        info['failure_reason'] = 'zero_time_variance'
        return None, info
    slope = float(np.dot(centered_t, voltage - voltage.mean()) / denominator)

    features = np.array([
        float(voltage.mean()),
        float(np.median(voltage)),
        duration,
        float(np.mean(np.abs(current))),
        float(qt) * QT_SCALE,
        slope,
    ], dtype=np.float64)
    if not np.all(np.isfinite(features)):
        info['failure_reason'] = 'nonfinite_feature'
        return None, info
    return features, info


def records_in_pickle(obj, stem):
    if not isinstance(obj, dict):
        return []
    if {'rul', 'dq', 'data'}.issubset(obj.keys()):
        return [(stem, obj)]
    return [
        (str(k), v) for k, v in obj.items()
        if isinstance(v, dict) and {'rul', 'dq', 'data'}.issubset(v.keys())
    ]


def load_hust_manuscript(hust_path=HUST_PATH):
    root = Path(hust_path)
    if not root.exists():
        raise FileNotFoundError(f'HUST path not found: {root}\nCandidates: {HUST_CANDIDATE_PATHS}')
    files = sorted(root.rglob('*.pkl'), key=lambda p: natural_key(str(p.relative_to(root))))
    if not files:
        raise FileNotFoundError(f'No .pkl files under {root}')
    print(f'Found {len(files)} candidate HUST pkl files')

    dataset = {}
    unsafe_phase_cycles = 0
    sample_index_cycles = 0
    failures = []

    for file in files:
        rel = str(file.relative_to(root))
        try:
            with file.open('rb') as handle:
                obj = pickle.load(handle)
        except Exception as exc:
            failures.append((rel, None, f'pickle_error:{exc!r}'))
            continue

        for cid, record in records_in_pickle(obj, file.stem):
            if cid in dataset:
                raise RuntimeError(f'Duplicate HUST cell ID {cid!r} in {rel}')
            dq, data = record.get('dq'), record.get('data')
            if not isinstance(dq, dict) or not isinstance(data, dict):
                failures.append((rel, cid, 'dq_or_data_not_dict'))
                continue

            shared = sorted(set(dq).intersection(data), key=natural_key)
            valid_candidates = []
            for cycle_id in shared:
                q = to_scalar(dq[cycle_id])
                if not np.isfinite(q) or q <= 0:
                    continue
                feature, info = extract_cycle_raw(data[cycle_id], q)
                if str(info.get('phase_policy', '')).startswith('unverified'):
                    unsafe_phase_cycles += 1
                if info.get('used_sample_index_time'):
                    sample_index_cycles += 1
                if feature is None:
                    continue
                valid_candidates.append((str(cycle_id), feature, q))

            if len(valid_candidates) <= WARMUP:
                failures.append((rel, cid, f'too_few_feature_valid_cycles:{len(valid_candidates)}'))
                continue

            # Verbatim v2 rule: Q0 = median of first five fully feature-valid capacities.
            q0 = float(np.median([item[2] for item in valid_candidates[:WARMUP]]))
            if not np.isfinite(q0) or q0 <= 0:
                failures.append((rel, cid, 'invalid_q0'))
                continue

            kept = []
            for cycle_id, feature, q in valid_candidates:
                soh = float(q / q0)
                if np.isfinite(soh) and 0.0 < soh < 1.5:
                    kept.append((cycle_id, feature, soh))
            if len(kept) <= WARMUP:
                failures.append((rel, cid, f'too_few_target_valid_cycles:{len(kept)}'))
                continue

            dataset[str(cid)] = {
                'features': np.vstack([x[1] for x in kept]).astype(np.float64),
                'soh': np.asarray([x[2] for x in kept], dtype=np.float64),
                'cycle_ids': np.asarray([x[0] for x in kept], dtype=object),
                'q0': q0,
            }

    if failures:
        print(f'{len(failures)} pkl/cell-level failures (first 10):')
        for rel, cid, reason in failures[:10]:
            print('  ', rel, cid, reason)
    if unsafe_phase_cycles:
        raise RuntimeError(
            f'{unsafe_phase_cycles} cycles used an unverified discharge-phase policy. '
            f'Set COLUMN_OVERRIDES["phase"] after inspecting the raw dataframes.'
        )
    if sample_index_cycles:
        raise RuntimeError(f'{sample_index_cycles} cycles fell back to sample-index time.')
    if failures:
        raise RuntimeError(
            f'{len(failures)} pkl/cell-level failures remain; resolve them before running '
            f'(the v2 audit run completed with zero failures on this data).'
        )

    return dict(sorted(dataset.items(), key=lambda item: natural_key(item[0])))


hust_dataset = load_hust_manuscript(HUST_PATH)
hust_cells = sorted(hust_dataset.keys())   # lexicographic: identical to v2 split ordering
N_INPUTS = int(hust_dataset[hust_cells[0]]['features'].shape[1])

print(f'✅ HUST loaded: {len(hust_cells)} cells, N_INPUTS={N_INPUTS}')
lengths = [len(hust_dataset[c]['soh']) for c in hust_cells]
print('Cycle length stats: min/median/max =', int(np.min(lengths)), int(np.median(lengths)), int(np.max(lengths)))
_sample = hust_cells[0]
print('Sample cell:', _sample, hust_dataset[_sample]['features'].shape, hust_dataset[_sample]['soh'].shape)
print('Feature order:', WITH_QT_FEATURE_NAMES)
print('Sample SOH first/last:', hust_dataset[_sample]['soh'][:3], hust_dataset[_sample]['soh'][-3:])

assert N_INPUTS == EXPECTED_INPUT_DIM == 6
assert all(v['features'].shape[1] == 6 for v in hust_dataset.values())

actual_sorted = sorted(hust_cells, key=natural_key)
expected_sorted = sorted(EXPECTED_CELL_IDS, key=natural_key)
if actual_sorted != expected_sorted:
    missing = sorted(set(EXPECTED_CELL_IDS) - set(hust_cells), key=natural_key)
    unexpected = sorted(set(hust_cells) - set(EXPECTED_CELL_IDS), key=natural_key)
    raise RuntimeError(
        f'Cell universe differs from the frozen v2 audit list. '
        f'Missing={missing}; unexpected={unexpected}. '
        f'Splits would not match v2 — do not proceed.'
    )
print('✅ Cell universe matches the frozen v2 audit list (77 cells)')


Found 77 candidate HUST pkl files
✅ HUST loaded: 77 cells, N_INPUTS=6
Cycle length stats: min/median/max = 1142 1875 2689
Sample cell: 1-1 (1504, 6) (1504,)
Feature order: ['voltage_mean', 'voltage_median', 'duration', 'abs_current_mean', 'Qt', 'voltage_slope']
Sample SOH first/last: [1.00152882 1.00052222 1.        ] [0.75140736 0.75088214 0.75088303]
✅ Cell universe matches the frozen v2 audit list (77 cells)


In [ ]:
# ================================================================
# 5. Deterministic splits, normalization, per-cell warmup, feature cache
# ================================================================
def get_hust_split(cells, seed):
    """Cell-disjoint 60/20/20 split, fully determined by (sorted cell list, seed)."""
    cells = list(cells)
    n_cells = len(cells)
    n_train = int(n_cells * 0.60)
    n_valid = int(n_cells * 0.20)
    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(cells).tolist()
    return (
        shuffled[:n_train],
        shuffled[n_train:n_train + n_valid],
        shuffled[n_train + n_valid:],
    )


def _find_prior_split_manifests(seed):
    """Collect split manifests for this seed saved by earlier parallel copies."""
    paths, payloads = [], []
    for exp_id in [*PRIOR_EXP_IDS, EXP_ID]:
        path = os.path.join(
            PROJECT_ROOT, 'results', exp_id, 'splits', f'HUST_seed{seed}_split.json'
        )
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as handle:
                payloads.append(json.load(handle))
            paths.append(path)
    return paths, payloads


def resolve_hust_split(seed):
    """Generate the deterministic split and verify it matches any saved manifest."""
    train_ids, valid_ids, test_ids = get_hust_split(hust_cells, seed)

    prior_paths, prior_payloads = _find_prior_split_manifests(seed)
    for path, payload in zip(prior_paths, prior_payloads):
        for key, ids in [
            ('train_cells', train_ids),
            ('valid_cells', valid_ids),
            ('test_cells', test_ids),
        ]:
            if list(payload.get(key, [])) != list(ids):
                raise AssertionError(
                    f'Split mismatch for seed {seed} against saved manifest {path}. '
                    f'The sorted HUST cell list must be identical across all copies.'
                )

    assert not (set(train_ids) & set(valid_ids))
    assert not (set(train_ids) & set(test_ids))
    assert not (set(valid_ids) & set(test_ids))
    assert set(train_ids + valid_ids + test_ids) == set(hust_cells)

    return train_ids, valid_ids, test_ids, prior_paths


def save_split_manifest(seed, train_ids, valid_ids, test_ids, prior_paths):
    payload = {
        'protocol_tag': PROTOCOL_TAG,
        'dataset': 'HUST',
        'seed': int(seed),
        'feature_names': HUST_FEATURE_NAMES,
        'warmup_cycles_per_cell': int(WARMUP),
        'consistent_prior_manifests': list(prior_paths),
        'train_cells': list(train_ids),
        'valid_cells': list(valid_ids),
        'test_cells': list(test_ids),
        'counts': {
            'train_cells': len(train_ids),
            'valid_cells': len(valid_ids),
            'test_cells': len(test_ids),
        },
    }
    path = os.path.join(SPLIT_DIR, f'HUST_seed{seed}_split.json')
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)
    return path


def fit_train_raw_stats(train_ids):
    retained = []
    for cell_id in train_ids:
        x_cell = np.asarray(hust_dataset[cell_id]['features'], dtype=np.float64)
        if len(x_cell) <= WARMUP:
            raise ValueError(f'Cell {cell_id} has only {len(x_cell)} cycles.')
        retained.append(x_cell[WARMUP:])
    stacked = np.vstack(retained)
    return {
        'mean': stacked.mean(axis=0),
        'std': stacked.std(axis=0) + 1e-8,
    }


def prepare_normalized_cells(all_ids, raw_stats):
    normalized = {}
    for cell_id in all_ids:
        x_cell = np.asarray(hust_dataset[cell_id]['features'], dtype=np.float64)
        y_cell = np.asarray(hust_dataset[cell_id]['soh'], dtype=np.float32).reshape(-1)
        if len(x_cell) != len(y_cell):
            raise ValueError(f'Length mismatch for cell {cell_id}: X={len(x_cell)}, y={len(y_cell)}')
        normalized[cell_id] = {
            'X_full': normalize_raw_features(x_cell, raw_stats).astype(np.float32),
            'y_full': y_cell,
        }
    return normalized


def _feature_cache_path(model_label, seed):
    safe = model_label.replace('+', '_').replace(' ', '')
    return os.path.join(FEAT_DIR, f'{safe}_HUST_seed{seed}_{PROTOCOL_TAG}.npz')


def _save_feature_cache(path, payload):
    np.savez_compressed(
        path,
        Rtr=payload['Rtr'], ytr=payload['ytr'], ctr=payload['ctr'], itr=payload['itr'],
        Rv=payload['Rv'], yv=payload['yv'], cv=payload['cv'], iv=payload['iv'],
        Rte=payload['Rte'], yte=payload['yte'], cte=payload['cte'], ite=payload['ite'],
    )


def _load_feature_cache(path):
    z = np.load(path, allow_pickle=True)
    return {key: z[key] for key in z.files}


def generate_split_features(model, normalized_cells, cell_ids, split_name, verbose=True):
    features, labels, cell_labels, cycle_indices = [], [], [], []
    start = time.time()

    for index, cell_id in enumerate(cell_ids):
        x_full = normalized_cells[cell_id]['X_full']
        y_full = normalized_cells[cell_id]['y_full']

        reservoir_full = model.process_cell(x_full)
        reservoir_kept = reservoir_full[WARMUP:]
        y_kept = y_full[WARMUP:]
        idx_kept = np.arange(len(y_full), dtype=np.int32)[WARMUP:]

        if len(reservoir_kept) != len(y_kept):
            raise RuntimeError(f'Post-warmup mismatch for cell {cell_id}')

        features.append(reservoir_kept)
        labels.append(y_kept)
        cell_labels.extend([str(cell_id)] * len(y_kept))
        cycle_indices.append(idx_kept)

        if verbose and ((index + 1) % 5 == 0 or index + 1 == len(cell_ids)):
            print(
                f'      {model.model_name} {split_name}: '
                f'{index + 1}/{len(cell_ids)} cells, {time.time() - start:.1f}s'
            )

    return {
        'R': np.vstack(features).astype(np.float32),
        'y': np.concatenate(labels).astype(np.float32),
        'cell': np.asarray(cell_labels, dtype=object),
        'cycle': np.concatenate(cycle_indices).astype(np.int32),
    }


def generate_or_load_all_features(model, seed, normalized_cells, train_ids, valid_ids, test_ids):
    cache_path = _feature_cache_path(model.model_name, seed)
    if USE_FEATURE_CACHE and not REGENERATE_FEATURES and os.path.exists(cache_path):
        log(f'Loading {model.model_name} feature cache: {cache_path}')
        return _load_feature_cache(cache_path), 0.0, cache_path

    start = time.time()
    train = generate_split_features(model, normalized_cells, train_ids, 'train')
    valid = generate_split_features(model, normalized_cells, valid_ids, 'valid')
    test = generate_split_features(model, normalized_cells, test_ids, 'test')

    payload = {
        'Rtr': train['R'], 'ytr': train['y'], 'ctr': train['cell'], 'itr': train['cycle'],
        'Rv': valid['R'], 'yv': valid['y'], 'cv': valid['cell'], 'iv': valid['cycle'],
        'Rte': test['R'], 'yte': test['y'], 'cte': test['cell'], 'ite': test['cycle'],
    }
    elapsed = time.time() - start

    if USE_FEATURE_CACHE:
        _save_feature_cache(cache_path, payload)

    return payload, elapsed, cache_path


In [ ]:
# ================================================================
# 6. Shared Tanh CBR training and artifact saving
# ================================================================
class CBRReadout(nn.Module):
    def __init__(self, n_inputs=57, hidden_dims=CBR_HIDDEN_DIMS, dropout=CBR_DROPOUT):
        super().__init__()
        layers = []
        previous = n_inputs
        for hidden in hidden_dims:
            layers.append(nn.Linear(previous, hidden))
            layers.append(nn.Tanh())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = hidden
        layers.append(nn.Linear(previous, 1))
        self.net = nn.Sequential(*layers)
        self.n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _batched_predict(model, x_array, batch_size=4096):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(x_array), batch_size):
            batch = torch.as_tensor(x_array[start : start + batch_size], dtype=torch.float32, device=device)
            outputs.append(model(batch).detach().cpu().numpy())
    return np.concatenate(outputs)


def train_cbr_paired(
    model_label,
    seed,
    Rtr,
    ytr,
    Rv,
    yv,
    Rte,
    yte,
    test_cells,
    test_cycles,
):
    seed_everything(seed)

    feature_mean = Rtr.mean(axis=0).astype(np.float32)
    feature_std = (Rtr.std(axis=0) + 1e-8).astype(np.float32)

    X_train = ((Rtr - feature_mean) / feature_std).astype(np.float32)
    X_valid = ((Rv - feature_mean) / feature_std).astype(np.float32)
    X_test = ((Rte - feature_mean) / feature_std).astype(np.float32)

    train_tensor = torch.as_tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.as_tensor(ytr, dtype=torch.float32)
    train_dataset = TensorDataset(train_tensor, y_train_tensor)

    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=CBR_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )

    X_valid_t = torch.as_tensor(X_valid, dtype=torch.float32, device=device)
    y_valid_t = torch.as_tensor(yv, dtype=torch.float32, device=device)

    # Reset the seed immediately before model construction so both paired
    # models start from identical CBR weights for the same seed.
    seed_everything(seed)
    model = CBRReadout(n_inputs=Rtr.shape[1]).to(device)
    if model.n_trainable != EXPECTED_CBR_PARAMS:
        raise RuntimeError(f"Expected 125 CBR parameters, got {model.n_trainable}")

    optimizer = optim.Adam(model.parameters(), lr=CBR_LR, weight_decay=CBR_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=10, factor=0.5, min_lr=1e-6
    )
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = []
    train_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(CBR_EPOCHS):
        model.train()
        batch_losses = []

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            predictions = model(x_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CBR_GRAD_CLIP)
            optimizer.step()
            batch_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            valid_predictions = model(X_valid_t)
            val_loss = float(criterion(valid_predictions, y_valid_t).item())

        train_loss = float(np.mean(batch_losses))
        scheduler.step(val_loss)
        current_lr = float(optimizer.param_groups[0]["lr"])

        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "valid_loss": val_loss,
                "learning_rate": current_lr,
            }
        )

        if val_loss < best_val_loss - 1e-12:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CBR_PATIENCE:
            break

    train_seconds = time.time() - train_start

    if best_state is None:
        raise RuntimeError("CBR training did not produce a checkpoint.")
    model.load_state_dict(best_state)

    inference_start = time.time()
    test_predictions = _batched_predict(model, X_test)
    inference_seconds = time.time() - inference_start
    metrics = compute_metrics(yte, test_predictions)

    safe_model = model_label.replace("+", "_").replace(" ", "")
    checkpoint_path = os.path.join(CKPT_DIR, f"{safe_model}_HUST_seed{seed}.pt")
    history_path = os.path.join(HIST_DIR, f"{safe_model}_HUST_seed{seed}_history.csv")
    prediction_path = os.path.join(PRED_DIR, f"{safe_model}_HUST_seed{seed}_predictions.csv")

    torch.save(
        {
            "protocol_tag": PROTOCOL_TAG,
            "model_label": model_label,
            "seed": int(seed),
            "state_dict": model.state_dict(),
            "feature_mean": feature_mean,
            "feature_std": feature_std,
            "architecture": [57, 2, 2, 1],
            "activation": "Tanh",
            "dropout": 0.0,
            "n_trainable": int(model.n_trainable),
        },
        checkpoint_path,
    )

    pd.DataFrame(history).to_csv(history_path, index=False)
    pd.DataFrame(
        {
            "cell_id": test_cells.astype(str),
            "cycle_index_zero_based": test_cycles.astype(int),
            "y_true": yte.astype(float),
            "y_pred": test_predictions.astype(float),
            "absolute_error": np.abs(yte - test_predictions).astype(float),
        }
    ).to_csv(prediction_path, index=False)

    peak_cuda_mb = (
        float(torch.cuda.max_memory_allocated(device) / (1024 ** 2))
        if torch.cuda.is_available()
        else 0.0
    )

    training_info = {
        "best_valid_loss": float(best_val_loss),
        "epochs_run": int(len(history)),
        "train_seconds": float(train_seconds),
        "inference_seconds": float(inference_seconds),
        "rss_mb_after_training": current_rss_mb(),
        "peak_cuda_mb": peak_cuda_mb,
        "checkpoint_path": checkpoint_path,
        "history_path": history_path,
        "prediction_path": prediction_path,
        "params": int(model.n_trainable),
    }
    return metrics, training_info


In [ ]:
# ================================================================
# 7. Protocol validation before expensive feature generation
# ================================================================
if len(hust_cells) != 77 or N_INPUTS != 6:
    raise RuntimeError(
        f'Expected 77 HUST cells and 6 inputs, got {len(hust_cells)} cells and {N_INPUTS} inputs.'
    )

cqrf_check = PairedReservoir10q(n_inputs=N_INPUTS, recurrent=False)
qrr_check = PairedReservoir10q(n_inputs=N_INPUTS, recurrent=True)

for reservoir_check in (cqrf_check, qrr_check):
    assert reservoir_check.n_outputs == 57
    assert reservoir_check.n_qubits == 10
    assert reservoir_check.memory_wires == [0, 1, 2, 3]
    assert reservoir_check.proc_wires == [4, 5, 6, 7, 8, 9]
    assert len(reservoir_check.proc_pairs) == 15
    assert len(reservoir_check.cross_pairs) == 24
    assert reservoir_check.input_projection.shape == (6, 6)

assert np.array_equal(cqrf_check.reservoir_params, qrr_check.reservoir_params)
assert np.array_equal(cqrf_check.input_projection, qrr_check.input_projection)
assert np.array_equal(cqrf_check.memory_rotation, qrr_check.memory_rotation)

rng_check = np.random.RandomState(RESERVOIR_SEED)
expected_reservoir_params = rng_check.uniform(
    0, 2 * np.pi, size=(N_RESERVOIR_LAYERS, N_MEMORY + N_PROCESSOR, 2)
)
expected_projection = rng_check.randn(N_INPUTS, N_PROCESSOR) * 0.5
expected_memory_rotation = rng_check.uniform(0, 2 * np.pi, size=(N_MEMORY, 2))

assert np.array_equal(cqrf_check.reservoir_params, expected_reservoir_params)
assert np.array_equal(cqrf_check.input_projection, expected_projection)
assert np.array_equal(cqrf_check.memory_rotation, expected_memory_rotation)

cbr_check = CBRReadout()
assert cbr_check.n_trainable == 125
assert sum(isinstance(layer, nn.Tanh) for layer in cbr_check.net) == 2

synthetic = np.random.RandomState(0).uniform(-1, 1, size=(4, N_INPUTS)).astype(np.float32)
cqrf_states = cqrf_check.process_cell(synthetic)
qrr_states = qrr_check.process_cell(synthetic)
assert cqrf_states.shape == qrr_states.shape == (4, 57)
assert np.all(np.isfinite(cqrf_states)) and np.all(np.isfinite(qrr_states))
assert np.allclose(cqrf_states[0], qrr_states[0], atol=1e-10)
assert not np.allclose(cqrf_states[1:], qrr_states[1:], atol=1e-6)
assert np.allclose(cqrf_check.process_cell(synthetic), cqrf_states, atol=1e-10)

for seed in ACTIVE_SEEDS:
    tr, va, te, prior = resolve_hust_split(seed)
    assert (len(tr), len(va), len(te)) == (46, 15, 16)
    print(
        f'Seed {seed}: split 46/15/16 OK; '
        f'consistent prior manifests (incl. v2)={len(prior)}'
    )

del cqrf_check, qrr_check, cbr_check
gc.collect()

print('✅ Paired protocol validation passed')
print('Input dimension: 6 (manuscript Eq. 12)')
print('Reservoir output dimension: 57')
print('CBR parameters: 125')
print('Models per seed:', MODEL_LABELS)


Seed 8: split 46/15/16 OK; consistent prior manifests (incl. v2)=1
Seed 9: split 46/15/16 OK; consistent prior manifests (incl. v2)=1
✅ Paired protocol validation passed
Input dimension: 6 (manuscript Eq. 12)
Reservoir output dimension: 57
CBR parameters: 125
Models per seed: ['CQRF+CBR', 'QRR+CBR']


In [ ]:
# ================================================================
# 8. Paired CQRF / QRR seed runner (skips seeds completed by v2)
# ================================================================
MASTER_RESULTS_PATH = os.path.join(RES_DIR, 'HUST_paired_run_metrics.csv')


def _load_metric_frames(exp_ids):
    frames = []
    for exp_id in exp_ids:
        for filename in PRIOR_METRIC_FILENAMES:
            path = os.path.join(PROJECT_ROOT, 'results', exp_id, filename)
            if os.path.exists(path):
                frame = pd.read_csv(path)
                frame['_source_metrics'] = path
                frames.append(frame)
    return frames


def _completed_keys():
    keys = set()
    for frame in _load_metric_frames([EXP_ID, *PRIOR_EXP_IDS]):
        needed = {'protocol_tag', 'seed', 'model'}
        if not needed.issubset(frame.columns):
            continue
        for _, row in frame.iterrows():
            keys.add((str(row['protocol_tag']), int(row['seed']), str(row['model'])))
    return keys


def _load_existing_results():
    if os.path.exists(MASTER_RESULTS_PATH):
        return pd.read_csv(MASTER_RESULTS_PATH)
    return pd.DataFrame()


existing_results = _load_existing_results()
completed_keys = _completed_keys() if SKIP_COMPLETED else set()
new_rows = []

for seed in ACTIVE_SEEDS:
    log('=' * 90)
    log(f'HUST paired seed {seed} (manuscript features): starting')
    seed_start = time.time()

    train_ids, valid_ids, test_ids, prior_split_paths = resolve_hust_split(seed)
    split_path = save_split_manifest(seed, train_ids, valid_ids, test_ids, prior_split_paths)

    pending_models = [
        label for label in MODEL_LABELS
        if (PROTOCOL_TAG, seed, label) not in completed_keys
    ]
    if not pending_models:
        log(f'Seed {seed}: both models already complete (here or in a prior/v2 run); skipping.')
        continue

    raw_stats = fit_train_raw_stats(train_ids)
    normalized_cells = prepare_normalized_cells(
        train_ids + valid_ids + test_ids,
        raw_stats,
    )

    for model_label in pending_models:
        recurrent = model_label == MODEL_LABEL_QRR
        reservoir = PairedReservoir10q(
            n_inputs=N_INPUTS,
            recurrent=recurrent,
            seed=RESERVOIR_SEED,
            coupling_strength=COUPLING_STRENGTH,
        )
        assert reservoir.model_name == model_label

        payload, feature_seconds, cache_path = generate_or_load_all_features(
            reservoir,
            seed,
            normalized_cells,
            train_ids,
            valid_ids,
            test_ids,
        )

        assert payload['Rtr'].shape[1] == 57
        assert payload['Rv'].shape[1] == 57
        assert payload['Rte'].shape[1] == 57

        log(f'Seed {seed} {model_label}: training shared Tanh CBR')

        metrics, train_info = train_cbr_paired(
            model_label=model_label,
            seed=seed,
            Rtr=payload['Rtr'],
            ytr=payload['ytr'],
            Rv=payload['Rv'],
            yv=payload['yv'],
            Rte=payload['Rte'],
            yte=payload['yte'],
            test_cells=payload['cte'],
            test_cycles=payload['ite'],
        )

        row = {
            'protocol_tag': PROTOCOL_TAG,
            'dataset': 'HUST',
            'seed': int(seed),
            'model': model_label,
            'feature_setting': 'With-Qt',
            'recurrent_memory': bool(recurrent),
            'raw_input_dim': int(N_INPUTS),
            'feature_names': json.dumps(WITH_QT_FEATURE_NAMES),
            'activation': 'Tanh',
            'readout_architecture': '57-2-2-1',
            'params': int(train_info['params']),
            'warmup_per_cell': int(WARMUP),
            'train_cells': len(train_ids),
            'valid_cells': len(valid_ids),
            'test_cells': len(test_ids),
            'n_train_cycles': int(len(payload['ytr'])),
            'n_valid_cycles': int(len(payload['yv'])),
            'n_test_cycles': int(len(payload['yte'])),
            'feature_seconds': float(feature_seconds),
            'train_seconds': float(train_info['train_seconds']),
            'inference_seconds': float(train_info['inference_seconds']),
            'epochs_run': int(train_info['epochs_run']),
            'best_valid_loss': float(train_info['best_valid_loss']),
            'split_manifest': split_path,
            'feature_cache': cache_path,
            'checkpoint': train_info['checkpoint_path'],
            'predictions': train_info['prediction_path'],
            'history': train_info['history_path'],
            **metrics,
        }
        new_rows.append(row)
        completed_keys.add((PROTOCOL_TAG, seed, model_label))

        log(
            f"Seed {seed} {model_label}: "
            f"MAPE={metrics['MAPE']:.6f}, RMSE={metrics['RMSE']:.6f}, "
            f"score={metrics['DerivedScore']:.4f}%"
        )

        combined = pd.concat(
            [existing_results, pd.DataFrame(new_rows)],
            ignore_index=True,
        )
        combined = combined.drop_duplicates(
            subset=['protocol_tag', 'dataset', 'seed', 'model'],
            keep='last',
        ).sort_values(['seed', 'model'])
        combined.to_csv(MASTER_RESULTS_PATH, index=False)

        del reservoir, payload
        gc.collect()

    log(f'HUST paired seed {seed}: finished in {time.time() - seed_start:.1f}s')

    del normalized_cells
    gc.collect()

print('✅ Active paired seed run complete')
print('Metrics:', MASTER_RESULTS_PATH)


[2026-07-20 05:58:32] ==========================================================================================
[2026-07-20 05:58:32] HUST paired seed 8 (manuscript features): starting
      CQRF+CBR train: 5/46 cells, 306.6s
      CQRF+CBR train: 10/46 cells, 692.7s
      CQRF+CBR train: 15/46 cells, 982.1s
      CQRF+CBR train: 20/46 cells, 1305.2s
      CQRF+CBR train: 25/46 cells, 1633.5s
      CQRF+CBR train: 30/46 cells, 1961.2s
      CQRF+CBR train: 35/46 cells, 2278.1s
      CQRF+CBR train: 40/46 cells, 2620.5s
      CQRF+CBR train: 45/46 cells, 2921.2s
      CQRF+CBR train: 46/46 cells, 2979.6s
      CQRF+CBR valid: 5/15 cells, 323.5s
      CQRF+CBR valid: 10/15 cells, 658.1s
      CQRF+CBR valid: 15/15 cells, 1018.9s
      CQRF+CBR test: 5/16 cells, 271.7s
      CQRF+CBR test: 10/16 cells, 598.1s
      CQRF+CBR test: 15/16 cells, 899.8s
      CQRF+CBR test: 16/16 cells, 969.7s
[2026-07-20 07:21:22] Seed 8 CQRF+CBR: training shared Tanh CBR
[2026-07-20 07:25:25] Seed 8 CQRF+C

In [ ]:
# ================================================================
# 9. Merged 10-seed summary (this notebook + v2 + parallel copies)
# ================================================================
metric_frames = _load_metric_frames([EXP_ID, *PRIOR_EXP_IDS])
if not metric_frames:
    raise RuntimeError('No HUST paired metrics CSV found yet.')

all_df = pd.concat(metric_frames, ignore_index=True)
all_df = all_df[
    (all_df['protocol_tag'] == PROTOCOL_TAG)
    & (all_df['dataset'] == 'HUST')
    & (all_df['model'].isin(MODEL_LABELS))
].copy()
all_df['seed'] = all_df['seed'].astype(int)
all_df = (
    all_df.sort_values('seed')
    .drop_duplicates(subset=['seed', 'model'], keep='last')
)

print('Rows merged per source:')
print(all_df.groupby('_source_metrics').size())

summary_rows = []
for model_label in MODEL_LABELS:
    model_df = all_df[all_df['model'] == model_label]
    if model_df.empty:
        continue
    summary_rows.append({
        'model': model_label,
        'n_seeds': int(model_df['seed'].nunique()),
        'MAPE_mean': float(model_df['MAPE'].mean()),
        'MAPE_std': float(model_df['MAPE'].std(ddof=1)) if len(model_df) > 1 else np.nan,
        'RMSE_mean': float(model_df['RMSE'].mean()),
        'RMSE_std': float(model_df['RMSE'].std(ddof=1)) if len(model_df) > 1 else np.nan,
        'score_mean': float(model_df['DerivedScore'].mean()),
        'score_std': float(model_df['DerivedScore'].std(ddof=1)) if len(model_df) > 1 else np.nan,
    })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(os.path.join(RES_DIR, 'HUST_paired_summary_merged.csv'), index=False)

pivot = all_df.pivot(index='seed', columns='model', values='MAPE').dropna()
if pivot.empty or MODEL_LABEL_CQRF not in pivot or MODEL_LABEL_QRR not in pivot:
    print('No overlapping seeds between CQRF and QRR yet.')
else:
    paired = pivot.rename(columns={
        MODEL_LABEL_CQRF: 'MAPE_CQRF',
        MODEL_LABEL_QRR: 'MAPE_QRR',
    }).reset_index()
    paired['MAPE_delta_QRR_minus_CQRF'] = paired['MAPE_QRR'] - paired['MAPE_CQRF']
    paired['relative_MAPE_change_pct'] = (
        paired['MAPE_delta_QRR_minus_CQRF'] / paired['MAPE_CQRF'] * 100.0
    )

    comparison_path = os.path.join(RES_DIR, 'HUST_CQRF_VS_QRR_paired_merged.csv')
    paired.to_csv(comparison_path, index=False)
    display(paired)

    print('Paired seeds available:', len(paired))
    print('QRR wins:', int((paired['MAPE_QRR'] < paired['MAPE_CQRF']).sum()), 'of', len(paired))
    print('Mean QRR minus CQRF MAPE:', paired['MAPE_delta_QRR_minus_CQRF'].mean())
    print('Paired comparison saved:', comparison_path)

    if len(paired) >= 2 and not np.allclose(paired['MAPE_delta_QRR_minus_CQRF'], 0):
        stat, p_two = wilcoxon(
            paired['MAPE_QRR'],
            paired['MAPE_CQRF'],
            alternative='two-sided',
            zero_method='wilcox',
        )
        note = '' if len(paired) == 10 else f' (PARTIAL: {len(paired)}/10 seeds)'
        print(f'Wilcoxon two-sided QRR vs CQRF: statistic={stat:.6f}, p={p_two:.6g}{note}')


Rows merged per source:
_source_metrics
/content/drive/MyDrive/HB/QRC_Battery_SOH/results/EXP-HUST-CQRF-QRR-CBR-TANH-PAIRED-10SEEDS-MIT-STRICT-REFERENCE-v2/HUST_per_seed_metrics.csv    3
/content/drive/MyDrive/HB/QRC_Battery_SOH/results/v3_C89_HUST_CQRF_QRR_CBR_PAIRED_MANUSCRIPT6_v1/HUST_paired_run_metrics.csv                     4
dtype: int64


,model,n_seeds,MAPE_mean,MAPE_std,RMSE_mean,RMSE_std,score_mean,score_std
0,CQRF+CBR,4,0.021233,0.004079,0.024125,0.003795,97.876697,0.407940
1,QRR+CBR,3,0.024011,0.004278,0.028457,0.006273,97.598930,0.427785


model,seed,MAPE_CQRF,MAPE_QRR,MAPE_delta_QRR_minus_CQRF,relative_MAPE_change_pct
0,0,0.019026,0.024935,0.005909,31.058228
1,8,0.019910,0.027751,0.007840,39.378198
2,9,0.018692,0.019346,0.000654,3.497876


Paired seeds available: 3
QRR wins: 0 of 3
Mean QRR minus CQRF MAPE: 0.004801112291833036
Paired comparison saved: /content/drive/MyDrive/HB/QRC_Battery_SOH/results/v3_C89_HUST_CQRF_QRR_CBR_PAIRED_MANUSCRIPT6_v1/HUST_CQRF_VS_QRR_paired_merged.csv
Wilcoxon two-sided QRR vs CQRF: statistic=0.000000, p=0.25 (PARTIAL: 3/10 seeds)


## Execution sequence

1. Run through **Protocol validation** with `RUN_MODE = "SMOKE"`.
2. Confirm:
   - `HUST loaded: 77 cells` and the frozen v2 cell-ID list matches exactly;
   - `N_INPUTS = 6` with the manuscript Eq. 12 feature order and Qt at index 4;
   - splits are 46/15/16 and every seed reports consistency with the saved v2 manifests;
   - CQRF and QRR share identical frozen parameters, agree on the first synthetic cycle, and diverge afterwards;
   - CBR parameter count is 125.
3. Set `RUN_MODE = "FULL"` and run the seed runner. Seeds already completed by the v2 notebook (listed in `PRIOR_EXP_IDS`) are skipped, so point `SEEDS` at the seeds v2 has **not** finished.
4. For parallel copies, change only `EXP_ID` and `SEEDS`; add every copy plus the v2 experiment ID to `PRIOR_EXP_IDS` in the copy used for the final summary.
5. The final cell merges this notebook's rows with v2's CQRF/QRR rows (same `PROTOCOL_TAG`), prints the per-seed paired table, and runs the Wilcoxon test — reported as PARTIAL until all 10 paired seeds exist.

**Compatibility warning:** merging with v2 rows is valid only because the extraction, cell universe, splits, reservoir, warmup, scaler, and CBR here are verbatim v2. If you change any of those, also change `PROTOCOL_TAG` so the rows can no longer be silently mixed. Feature caches from the old (pre-correction) 346 v1 notebook are incompatible; this notebook uses a fresh `EXP_ID`, so they are never touched.
